# **Build a Domain-Specific AI Assistant using Unsloth**
## **Insurance AI Assist - STAGE 2**

In [ ]:
print("3 -> 2 --> 1 ---> GO!🚀\nStarting with the Instruction Fine-Tuning (Stage 2 Module)")

# Load Stage 1 Model from HF Repository
#           ↓
# Load Instruction Dataset from HF Dataset Repository
#           ↓
# Apply Alpaca Formatting
#           ↓
# Supervised Fine-Tuning (SFT)
#           ↓
# Save Stage 2 Model (Colab)
#           ↓
# Upload Stage 2 Model to HF Repository
#           ↓
# Download Latest Stage 1 Evaluation Report from HF Repository
#           ↓
# Generate Stage 2 Evaluation Report
#           ↓
# Append SFT Answers to Stage 1 Evaluation Results
#           ↓
# Save stage2_evaluation_report_<timestamp>.md
#           ↓
# Upload Report to Stage 2 HF Repository

3 -> 2 --> 1 ---> GO!🚀
Starting with the Instruction Fine-Tuning (Stage 2 Module)


Install Dependencies

In [ ]:
# =====================================================
# Install Project Dependencies
# =====================================================
!pip install -r ../requirements.txt

# =====================================================
# !pip -q install unsloth
# !pip -q install transformers==4.56.2
# !pip -q install --no-deps trl==0.22.2
# !pip -q install datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 24.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.2 MB/s eta 0:00:00:00:01
  

IDE Configuration

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

Hugging Face Login

In [ ]:
# =====================================================
# Hugging Face Authentication & Validation
# =====================================================

from google.colab import userdata
from huggingface_hub import (
    login,
    HfApi,
)

HF_REPO_NAME = "Pzazz55"

HF_REPOS = {
    "dataset": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-dataset",
    "stage1": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage1",
    "stage2": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage2",
    "stage3": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage3",
}

# =====================================================
# Load HF Token
# =====================================================

# HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        """
❌ Hugging Face token not found.

Please add HF_TOKEN to Google Colab Secrets.

Colab:
Left Sidebar -> Secrets -> Add New Secret

Name:
HF_TOKEN

Value:
Your Hugging Face Access Token
"""
    )

# =====================================================
# Login
# =====================================================

try:

    login(
        token=HF_TOKEN,
        add_to_git_credential=True
    )

    api = HfApi(token=HF_TOKEN)

    whoami = api.whoami()

    print(
        f"✅ Hugging Face Authenticated"
    )

    print(
        f"✅ Logged in as: {whoami['name']}"
    )

except Exception as e:

    raise RuntimeError(
        f"""
❌ Hugging Face authentication failed.

Please verify:

1. HF_TOKEN is valid.
2. HF_TOKEN has Read access.
3. HF_TOKEN has Write access.

Original Error:
{e}
"""
    )

# =====================================================
# Verify Repository Access
# =====================================================

print("\n🔍 Verifying Repository Access...")

# Dataset Repository
try:
    api.repo_info(
        repo_id=HF_REPOS["dataset"],
        repo_type="dataset"
    )

    print(
        f"✅ dataset: "
        f"{HF_REPOS['dataset']}"
    )

except Exception as e:

    raise RuntimeError(
        f"""
❌ Dataset repository validation failed.

Repository:
{HF_REPOS['dataset']}

Error:
{e}
"""
    )

# Model Repositories
for repo_key in [
    "stage1",
    "stage2",
    "stage3"
]:

    try:

        api.repo_info(
            repo_id=HF_REPOS[repo_key],
            repo_type="model"
        )

        print(
            f"✅ {repo_key}: "
            f"{HF_REPOS[repo_key]}"
        )

    except Exception as e:

        raise RuntimeError(
            f"""
❌ Model repository validation failed.

Repository:
{HF_REPOS[repo_key]}

Error:
{e}
"""
        )

print(
    "\n🚀 Hugging Face validation successful."
)

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
✅ Hugging Face Authenticated
✅ Logged in as: Pzazz55

🔍 Verifying Repository Access...
✅ dataset: Pzazz55/insurance-ai-assistant-finetuning-dataset
✅ stage1: Pzazz55/insurance-ai-assistant-finetuning-stage1
✅ stage2: Pzazz55/insurance-ai-assistant-finetuning-stage2
✅ stage3: Pzazz55/insurance-ai-assistant-finetuning-stage3

🚀 Hugging Face validation successful.


Imports & Configuration

In [ ]:
import os
import gc
import torch

from datasets import load_dataset

from unsloth import FastLanguageModel

from trl import (
    SFTTrainer,
    SFTConfig,
)

from huggingface_hub import (
    create_repo,
    upload_folder,
)

MAX_SEQ_LENGTH = 1024

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/reports", exist_ok=True)
os.makedirs("/content/outputs_sft", exist_ok=True)

# =====================================================
# Hugging Face Configuration
# =====================================================

HF_REPO_NAME = "Pzazz55"

HF_REPOS = {
    "dataset": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-dataset",
    "stage1": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage1",
    "stage2": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage2",
    "stage3": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage3",
}

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Create Stage 2 HF Repository

In [ ]:
create_repo(
    repo_id=HF_REPOS["stage2"],
    exist_ok=True,
)

print("✅ Stage 2 Repository Ready")

✅ Stage 2 Repository Ready


Load Stage 1 Model From HF

In [ ]:
print("Loading Stage 1 model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_REPOS["stage1"],
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print("✅ Stage 1 Model Loaded")

Loading Stage 1 model...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Stage 1 Model Loaded


Apply Additional LoRA For SFT

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("✅ SFT LoRA Attached")

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ SFT LoRA Attached


Alpaca Prompt Template

In [ ]:
ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{response}"""

Load Instruction Dataset

In [ ]:
from datasets import load_dataset

# Load only the specific .jsonl file directly using your HF Repo ID
dataset = load_dataset(
    # "Pzazz55/insurance-ai-assistant-finetuning-dataset",
    HF_REPOS["dataset"],
    data_files="insurance_instruction_dataset.jsonl"
)

# Access your data inside the 'train' split
print(f"✅ Successfully loaded specific file!")
print(f"Total records: {len(dataset['train'])}")

insurance_instruction_dataset.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ Successfully loaded specific file!
Total records: 120


Format Dataset

In [ ]:
EOS_TOKEN = tokenizer.eos_token

def format_examples(examples):

    formatted_texts = []

    for instruction, response in zip(
        examples["instruction"],
        examples["output"]
    ):

        formatted_texts.append(
            ALPACA_PROMPT.format(
                instruction=instruction.strip(),
                response=response.strip()
            ) + EOS_TOKEN
        )

    return {"text": formatted_texts}


dataset = dataset.map(
    format_examples,
    batched=True,
    desc="Formatting dataset using Alpaca template"
)

print("✅ Alpaca formatting applied")

Formatting dataset using Alpaca template:   0%|          | 0/120 [00:00<?, ? examples/s]

✅ Alpaca formatting applied


Validate Dataset

In [ ]:
print(dataset["train"][0]["text"])

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Explain auto insurance in insurance context.

### Response:
Auto insurance covers financial losses from vehicle accidents, theft, and damage. It typically includes liability, collision, and comprehensive coverage. It plays a key role in insurance operations, including policy administration, claims processing, risk assessment, and customer servicing. Understanding this concept helps policyholders and insurers make informed decisions.<|im_end|>


Configure SFT

In [ ]:
sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=30,
    learning_rate=2e-5,
    dataloader_num_workers=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    output_dir="outputs_sft",
    report_to="none",
    save_strategy="no",
)

Create Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    args=sft_config,
)

print("✅ SFT Trainer Initialized")
print(f"Training Records: {len(dataset['train'])}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/120 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/120 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
✅ SFT Trainer Initialized
Training Records: 120


Train

In [ ]:
trainer.train()
print("✅ Stage 2 Training Complete")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16 | Num Epochs = 8 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 9,232,384 of 1,552,946,688 (0.59% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,3.428700
10,3.340800
15,3.086700
20,2.846200
25,2.688200
30,2.602300


✅ Stage 2 Training Complete


In [ ]:

# =====================================================
# Clear Training Cache
# =====================================================

gc.collect()
torch.cuda.empty_cache()

print("✅ Training cache cleared")

print(
    f"GPU Allocated: "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"GPU Reserved: "
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

✅ Training cache cleared
GPU Allocated: 1.18 GB
GPU Reserved: 1.25 GB


Save Stage 2

In [ ]:
# =====================================================
# Save Model
# ===================================================

stage2_path = (
    "/content/models/stage2_sft"
)

model.save_pretrained_merged(
    stage2_path,
    tokenizer,
    save_method="merged_16bit",
)

print("✅ Stage 2 Saved")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/models/stage2_sft`: 100%|██████████| 1/1 [00:01<00:00,  1.95s/it]


Successfully copied all 1 files from cache to `/content/models/stage2_sft`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:22<00:00, 22.33s/it]


Unsloth: Merge process complete. Saved to `/content/models/stage2_sft`
✅ Stage 2 Saved


In [ ]:
# =====================================================
# Release Trainer Memory
# =====================================================

del trainer

gc.collect()
torch.cuda.empty_cache()

print("✅ Trainer removed from memory")

print(
    f"GPU Allocated: "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"GPU Reserved: "
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

✅ Trainer removed from memory
GPU Allocated: 1.16 GB
GPU Reserved: 1.21 GB


Upload Stage 2 To HF

In [ ]:
# =====================================================
# Clean HF Repository - Except /results directory
# =====================================================

import os

from huggingface_hub import (
    HfApi,
    upload_folder,
)

api = HfApi(token=HF_TOKEN)

repo_files = api.list_repo_files(
    repo_id=HF_REPOS["stage2"],
    repo_type="model",
)

files_to_delete = [
    file
    for file in repo_files
    if not file.startswith("results/")
]

if files_to_delete:

    api.delete_files(
        delete_patterns=files_to_delete,
        repo_id=HF_REPOS["stage2"],
        repo_type="model",
        commit_message=(
            "Cleanup old model artifacts before Stage 2 upload"
        ),
    )

    print(
        f"✅ Deleted {len(files_to_delete)} old repository files"
    )

else:

    print(
        "✅ No old model artifacts found"
    )

# =====================================================
# Upload Latest Stage 2 Model
# =====================================================

upload_folder(
    repo_id=HF_REPOS["stage2"],
    folder_path=stage2_path,
)

print(
    "✅ Stage 2 Model Uploaded"
)

✅ Deleted 11 old repository files


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Stage 2 Model Uploaded


Alpaca Inference Function

In [ ]:
def generate_answer(question):

    prompt = f"""Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            temperature=0.1,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    answer = text.split(
        "### Response:"
    )[-1].strip()

    return answer

In [ ]:
# =====================================================
# Switch To Inference Mode
# =====================================================

FastLanguageModel.for_inference(model)

model.eval()

torch.set_grad_enabled(False)

gc.collect()
torch.cuda.empty_cache()

print("✅ Inference mode enabled")

✅ Inference mode enabled


Build Stage 2 Evaluation Report

In [ ]:
# =====================================================
# Download Latest Stage 1 Evaluation Report
# =====================================================

import os
from datetime import datetime

from huggingface_hub import (
    HfApi,
    hf_hub_download,
)

api = HfApi(token=HF_TOKEN)

INPUT_REPO_ID = HF_REPOS["stage1"]
OUTPUT_REPO_ID = HF_REPOS["stage2"]

# =====================================================
# Find Latest Stage 1 Report
# =====================================================

files = api.list_repo_files(
    repo_id=INPUT_REPO_ID,
    repo_type="model",
)

report_files = [
    f
    for f in files
    if (
        f.startswith("results/")
        and "stage1_evaluation_report_" in f
        and f.endswith(".md")
    )
]

if not report_files:
    raise RuntimeError(
        "No Stage 1 evaluation reports found."
    )

latest_report = sorted(report_files)[-1]

# =====================================================
# Download Report
# =====================================================

local_stage1_report = hf_hub_download(
    repo_id=INPUT_REPO_ID,
    filename=latest_report,
    repo_type="model",
)

print(
    f"✅ Downloaded:\n{local_stage1_report}"
)

# =====================================================
# Read Report
# =====================================================

with open(
    local_stage1_report,
    "r",
    encoding="utf-8",
) as f:

    lines = f.readlines()

# =====================================================
# Parse Existing Table
# =====================================================

records = []

for line in lines:

    line = line.strip()

    if not line.startswith("|"):
        continue

    if (
        "Question" in line
        or "---" in line
        or ":---" in line
    ):
        continue

    cols = [
        c.strip()
        for c in line.split("|")[1:-1]
    ]

    if len(cols) < 4:
        continue

    records.append(
        (
            cols[0],  # Question No.
            cols[1],  # Question
            cols[2],  # Base Model Answer
            cols[3],  # Non-Instruction Model Answer
        )
    )

print(
    f"✅ Parsed {len(records)} questions"
)

if not records:
    raise RuntimeError(
        "No valid evaluation records found inside Stage 1 report."
    )

# =====================================================
# Generate Stage 2 Evaluation Report
# =====================================================

print(
    f"\n🚀 Evaluating SFT Model..."
    f"({len(records)} questions)"
)

markdown = [
    "# Stage 2 Evaluation Report",
    "",
    "| Question No. | Question | Base Model Answer | Non-Instruction Model Answer | Instruction Model Answer |",
    "|---|---|---|---|---|",
]

for idx, (
    question_no,
    question,
    base_answer,
    stage1_answer,
) in enumerate(records, start=1):

    print(
        f"[{idx}/{len(records)}] Evaluating..."
    )

    clean_question = (
        question
        .replace("<br>", "\n")
        .replace("<br/>", "\n")
    )

    ft_answer = generate_answer(
        clean_question
    )

    safe_question = (
        question
        .replace("|", "\\|")
    )

    safe_base_answer = (
        base_answer
        .replace("|", "\\|")
    )

    safe_stage1_answer = (
        stage1_answer
        .replace("|", "\\|")
    )

    safe_ft_answer = (
        ft_answer
        .replace("\n", "<br>")
        .replace("|", "\\|")
    )

    markdown.append(
        f"| {question_no} "
        f"| {safe_question} "
        f"| {safe_base_answer} "
        f"| {safe_stage1_answer} "
        f"| {safe_ft_answer} |"
    )

print(
    "✅ Stage 2 Evaluation Complete"
)

# =====================================================
# Save Report
# =====================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_report_path = (
    f"/content/reports/"
    f"stage2_evaluation_report_{timestamp}.md"
)

os.makedirs(
    "/content/reports",
    exist_ok=True,
)

with open(
    output_report_path,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        "\n".join(markdown)
    )

print(
    "\n✅ Stage 2 Report Saved"
)

print(
    output_report_path
)

✅ Downloaded:
/root/.cache/huggingface/hub/models--Pzazz55--insurance-ai-assistant-finetuning-stage1/snapshots/e9b9278f6b0580ff6c576d5194f1005d6445c4d5/results/stage1_evaluation_report_20260709_075341.md
✅ Parsed 10 questions

🚀 Evaluating SFT Model...(10 questions)
[1/10] Evaluating...
[2/10] Evaluating...
[3/10] Evaluating...
[4/10] Evaluating...
[5/10] Evaluating...
[6/10] Evaluating...
[7/10] Evaluating...
[8/10] Evaluating...
[9/10] Evaluating...
[10/10] Evaluating...
✅ Stage 2 Evaluation Complete

✅ Stage 2 Report Saved
/content/reports/stage2_evaluation_report_20260709_124941.md


Upload Report To HF

In [ ]:
# =====================================================
# Upload Reports
# =====================================================

uploaded_count = 0

if os.path.exists("/content/reports"):

    for file in os.listdir("/content/reports"):

        local_file = os.path.join(
            "/content/reports",
            file,
        )

        if not os.path.isfile(local_file):
            continue

        try:

            api.upload_file(
                path_or_fileobj=local_file,
                path_in_repo=f"results/{file}",
                repo_id=HF_REPOS["stage2"],
                repo_type="model",
            )

            uploaded_count += 1

            print(
                f"✅ Uploaded {file}"
            )

        except Exception as e:

            print(
                f"❌ Failed: {file}"
            )

            print(e)

print(
    f"\n✅ Upload Complete "
    f"({uploaded_count} file(s))"
)

print(
    f"https://huggingface.co/{HF_REPOS['stage2']}/tree/main/results"
)

✅ Uploaded stage2_evaluation_report_20260709_124941.md

✅ Upload Complete (1 file(s))
https://huggingface.co/Pzazz55/insurance-ai-assistant-finetuning-stage2/tree/main/results


Final Cleanup - Release Memory

In [ ]:
# =====================================================
# Final Cleanup
# =====================================================

del model

gc.collect()
torch.cuda.empty_cache()

print("✅ GPU memory fully released")

print(
    f"GPU Allocated: "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"GPU Reserved: "
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

✅ GPU memory fully released
GPU Allocated: 0.48 GB
GPU Reserved: 0.56 GB
